# 05 - Segmentation Configuration Selection

This notebook documents the exploratory study used to choose the ECG segmentation configuration before fixing the main baselines.


## Goal

Compare different values of:
- `window_sec`
- `overlap`

using the same feature engineering and the same baseline models, then choose a configuration for the main privacy-utility study.


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from config import OUTPUTS_TABLES_DIR

## Experimental Design

Two stages were run:

1. Pilot sweep on `1000` records with four configurations:
   - `w1_o0p5`
   - `w2_o0p5`
   - `w2_o0p75`
   - `w3_o0p5`
2. Focused comparison on `5000` records with the two strongest candidates:
   - `w2_o0p5`
   - `w3_o0p5`

Metrics were computed for both tasks:
- clinical utility
- linkability


In [3]:
pilot_results = pd.DataFrame([
    {"config": "w1_o0p5", "window_sec": 1.0, "overlap": 0.5, "record_count": 1000, "segment_count": 19000, "model": "LogisticRegression", "task": "utility", "f1_score": 0.645191, "balanced_accuracy": 0.805188, "roc_auc": 0.885043, "pr_auc": 0.691483},
    {"config": "w1_o0p5", "window_sec": 1.0, "overlap": 0.5, "record_count": 1000, "segment_count": 19000, "model": "XGBoost", "task": "utility", "f1_score": 0.652174, "balanced_accuracy": 0.771477, "roc_auc": 0.898723, "pr_auc": 0.725073},
    {"config": "w1_o0p5", "window_sec": 1.0, "overlap": 0.5, "record_count": 1000, "segment_count": 19000, "model": "LogisticRegression", "task": "linkability", "f1_score": 0.929204, "balanced_accuracy": 0.928000, "roc_auc": 0.976034, "pr_auc": 0.967663},
    {"config": "w1_o0p5", "window_sec": 1.0, "overlap": 0.5, "record_count": 1000, "segment_count": 19000, "model": "XGBoost", "task": "linkability", "f1_score": 0.955905, "balanced_accuracy": 0.956500, "roc_auc": 0.993140, "pr_auc": 0.993677},
    {"config": "w2_o0p5", "window_sec": 2.0, "overlap": 0.5, "record_count": 1000, "segment_count": 9000, "model": "LogisticRegression", "task": "utility", "f1_score": 0.669284, "balanced_accuracy": 0.823138, "roc_auc": 0.902474, "pr_auc": 0.731918},
    {"config": "w2_o0p5", "window_sec": 2.0, "overlap": 0.5, "record_count": 1000, "segment_count": 9000, "model": "XGBoost", "task": "utility", "f1_score": 0.668348, "balanced_accuracy": 0.783074, "roc_auc": 0.912600, "pr_auc": 0.730835},
    {"config": "w2_o0p5", "window_sec": 2.0, "overlap": 0.5, "record_count": 1000, "segment_count": 9000, "model": "LogisticRegression", "task": "linkability", "f1_score": 0.968680, "balanced_accuracy": 0.968750, "roc_auc": 0.991878, "pr_auc": 0.993067},
    {"config": "w2_o0p5", "window_sec": 2.0, "overlap": 0.5, "record_count": 1000, "segment_count": 9000, "model": "XGBoost", "task": "linkability", "f1_score": 0.973658, "balanced_accuracy": 0.974000, "roc_auc": 0.996946, "pr_auc": 0.997275},
    {"config": "w2_o0p75", "window_sec": 2.0, "overlap": 0.75, "record_count": 1000, "segment_count": 17000, "model": "LogisticRegression", "task": "utility", "f1_score": 0.681075, "balanced_accuracy": 0.830002, "roc_auc": 0.901907, "pr_auc": 0.733682},
    {"config": "w2_o0p75", "window_sec": 2.0, "overlap": 0.75, "record_count": 1000, "segment_count": 17000, "model": "XGBoost", "task": "utility", "f1_score": 0.671543, "balanced_accuracy": 0.785663, "roc_auc": 0.912571, "pr_auc": 0.738685},
    {"config": "w2_o0p75", "window_sec": 2.0, "overlap": 0.75, "record_count": 1000, "segment_count": 17000, "model": "LogisticRegression", "task": "linkability", "f1_score": 0.978916, "balanced_accuracy": 0.979000, "roc_auc": 0.996174, "pr_auc": 0.995827},
    {"config": "w2_o0p75", "window_sec": 2.0, "overlap": 0.75, "record_count": 1000, "segment_count": 17000, "model": "XGBoost", "task": "linkability", "f1_score": 0.978809, "balanced_accuracy": 0.979000, "roc_auc": 0.998264, "pr_auc": 0.998407},
    {"config": "w3_o0p5", "window_sec": 3.0, "overlap": 0.5, "record_count": 1000, "segment_count": 5000, "model": "LogisticRegression", "task": "utility", "f1_score": 0.690909, "balanced_accuracy": 0.835125, "roc_auc": 0.907590, "pr_auc": 0.780022},
    {"config": "w3_o0p5", "window_sec": 3.0, "overlap": 0.5, "record_count": 1000, "segment_count": 5000, "model": "XGBoost", "task": "utility", "f1_score": 0.678815, "balanced_accuracy": 0.789176, "roc_auc": 0.920487, "pr_auc": 0.751628},
    {"config": "w3_o0p5", "window_sec": 3.0, "overlap": 0.5, "record_count": 1000, "segment_count": 5000, "model": "LogisticRegression", "task": "linkability", "f1_score": 0.977330, "balanced_accuracy": 0.977500, "roc_auc": 0.995200, "pr_auc": 0.996618},
    {"config": "w3_o0p5", "window_sec": 3.0, "overlap": 0.5, "record_count": 1000, "segment_count": 5000, "model": "XGBoost", "task": "linkability", "f1_score": 0.982278, "balanced_accuracy": 0.982500, "roc_auc": 0.999125, "pr_auc": 0.999167},
])

pilot_results.sort_values(["task", "model", "config"]).reset_index(drop=True)


,config,window_sec,overlap,record_count,segment_count,model,task,f1_score,balanced_accuracy,roc_auc,pr_auc
0,w1_o0p5,1.0,0.50,1000,19000,LogisticRegression,linkability,0.929204,0.928000,0.976034,0.967663
1,w2_o0p5,2.0,0.50,1000,9000,LogisticRegression,linkability,0.968680,0.968750,0.991878,0.993067
2,w2_o0p75,2.0,0.75,1000,17000,LogisticRegression,linkability,0.978916,0.979000,0.996174,0.995827
3,w3_o0p5,3.0,0.50,1000,5000,LogisticRegression,linkability,0.977330,0.977500,0.995200,0.996618
4,w1_o0p5,1.0,0.50,1000,19000,XGBoost,linkability,0.955905,0.956500,0.993140,0.993677
5,w2_o0p5,2.0,0.50,1000,9000,XGBoost,linkability,0.973658,0.974000,0.996946,0.997275
6,w2_o0p75,2.0,0.75,1000,17000,XGBoost,linkability,0.978809,0.979000,0.998264,0.998407
7,w3_o0p5,3.0,0.50,1000,5000,XGBoost,linkability,0.982278,0.982500,0.999125,0.999167
8,w1_o0p5,1.0,0.50,1000,19000,LogisticRegression,utility,0.645191,0.805188,0.885043,0.691483
9,w2_o0p5,2.0,0.50,1000,9000,LogisticRegression,utility,0.669284,0.823138,0.902474,0.731918


## Pilot Reading

Main conclusions from the `1000`-record pilot:
- `w1_o0p5` underperformed on utility.
- `w2_o0p75` kept linkability very high without a clear utility advantage.
- `w3_o0p5` gave the best utility among the four tested configurations.
- `w2_o0p5` looked like the best intermediate candidate to compare against `w3_o0p5`.


In [4]:
final_candidates_results = pd.DataFrame([
    {"config": "w2_o0p5", "window_sec": 2.0, "overlap": 0.5, "record_count": 5000, "segment_count": 45000, "model": "LogisticRegression", "task": "utility", "f1_score": 0.723567, "balanced_accuracy": 0.854232, "roc_auc": 0.930469, "pr_auc": 0.794538},
    {"config": "w2_o0p5", "window_sec": 2.0, "overlap": 0.5, "record_count": 5000, "segment_count": 45000, "model": "XGBoost", "task": "utility", "f1_score": 0.723949, "balanced_accuracy": 0.807871, "roc_auc": 0.939952, "pr_auc": 0.821477},
    {"config": "w2_o0p5", "window_sec": 2.0, "overlap": 0.5, "record_count": 5000, "segment_count": 45000, "model": "LogisticRegression", "task": "linkability", "f1_score": 0.975488, "balanced_accuracy": 0.975500, "roc_auc": 0.996212, "pr_auc": 0.995470},
    {"config": "w2_o0p5", "window_sec": 2.0, "overlap": 0.5, "record_count": 5000, "segment_count": 45000, "model": "XGBoost", "task": "linkability", "f1_score": 0.987253, "balanced_accuracy": 0.987250, "roc_auc": 0.998898, "pr_auc": 0.998942},
    {"config": "w3_o0p5", "window_sec": 3.0, "overlap": 0.5, "record_count": 5000, "segment_count": 25000, "model": "LogisticRegression", "task": "utility", "f1_score": 0.737304, "balanced_accuracy": 0.863631, "roc_auc": 0.939301, "pr_auc": 0.825578},
    {"config": "w3_o0p5", "window_sec": 3.0, "overlap": 0.5, "record_count": 5000, "segment_count": 25000, "model": "XGBoost", "task": "utility", "f1_score": 0.754435, "balanced_accuracy": 0.827408, "roc_auc": 0.950988, "pr_auc": 0.852889},
    {"config": "w3_o0p5", "window_sec": 3.0, "overlap": 0.5, "record_count": 5000, "segment_count": 25000, "model": "LogisticRegression", "task": "linkability", "f1_score": 0.988978, "balanced_accuracy": 0.989000, "roc_auc": 0.999280, "pr_auc": 0.999274},
    {"config": "w3_o0p5", "window_sec": 3.0, "overlap": 0.5, "record_count": 5000, "segment_count": 25000, "model": "XGBoost", "task": "linkability", "f1_score": 0.988471, "balanced_accuracy": 0.988500, "roc_auc": 0.999357, "pr_auc": 0.999376},
])

final_candidates_results.sort_values(["task", "model", "config"]).reset_index(drop=True)


,config,window_sec,overlap,record_count,segment_count,model,task,f1_score,balanced_accuracy,roc_auc,pr_auc
0,w2_o0p5,2.0,0.5,5000,45000,LogisticRegression,linkability,0.975488,0.975500,0.996212,0.995470
1,w3_o0p5,3.0,0.5,5000,25000,LogisticRegression,linkability,0.988978,0.989000,0.999280,0.999274
2,w2_o0p5,2.0,0.5,5000,45000,XGBoost,linkability,0.987253,0.987250,0.998898,0.998942
3,w3_o0p5,3.0,0.5,5000,25000,XGBoost,linkability,0.988471,0.988500,0.999357,0.999376
4,w2_o0p5,2.0,0.5,5000,45000,LogisticRegression,utility,0.723567,0.854232,0.930469,0.794538
5,w3_o0p5,3.0,0.5,5000,25000,LogisticRegression,utility,0.737304,0.863631,0.939301,0.825578
6,w2_o0p5,2.0,0.5,5000,45000,XGBoost,utility,0.723949,0.807871,0.939952,0.821477
7,w3_o0p5,3.0,0.5,5000,25000,XGBoost,utility,0.754435,0.827408,0.950988,0.852889


In [5]:
utility_comparison = final_candidates_results[final_candidates_results["task"] == "utility"].copy()
linkability_comparison = final_candidates_results[final_candidates_results["task"] == "linkability"].copy()

print("Utility comparison")
display(utility_comparison.sort_values(["model", "config"]).reset_index(drop=True))

print("Linkability comparison")
display(linkability_comparison.sort_values(["model", "config"]).reset_index(drop=True))


Utility comparison


,config,window_sec,overlap,record_count,segment_count,model,task,f1_score,balanced_accuracy,roc_auc,pr_auc
0,w2_o0p5,2.0,0.5,5000,45000,LogisticRegression,utility,0.723567,0.854232,0.930469,0.794538
1,w3_o0p5,3.0,0.5,5000,25000,LogisticRegression,utility,0.737304,0.863631,0.939301,0.825578
2,w2_o0p5,2.0,0.5,5000,45000,XGBoost,utility,0.723949,0.807871,0.939952,0.821477
3,w3_o0p5,3.0,0.5,5000,25000,XGBoost,utility,0.754435,0.827408,0.950988,0.852889


Linkability comparison


,config,window_sec,overlap,record_count,segment_count,model,task,f1_score,balanced_accuracy,roc_auc,pr_auc
0,w2_o0p5,2.0,0.5,5000,45000,LogisticRegression,linkability,0.975488,0.97550,0.996212,0.995470
1,w3_o0p5,3.0,0.5,5000,25000,LogisticRegression,linkability,0.988978,0.98900,0.999280,0.999274
2,w2_o0p5,2.0,0.5,5000,45000,XGBoost,linkability,0.987253,0.98725,0.998898,0.998942
3,w3_o0p5,3.0,0.5,5000,25000,XGBoost,linkability,0.988471,0.98850,0.999357,0.999376


## Final Decision

Selected main configuration for the project:
- `w2_o0p5`
- `window_sec = 2.0`
- `overlap = 0.5`
- equivalent `step_sec = 1.0`

Rationale:
- it improves utility over the original `w2_o0p75` setup;
- it keeps the design simpler and less redundant than `75%` overlap;
- it is a more balanced choice for the privacy-utility trade-off than `w3_o0p5`.

Secondary comparison configuration:
- `w3_o0p5`

Why keep `w3_o0p5` as a comparison:
- it gives the strongest utility results;
- but it also pushes linkability even higher.


In [6]:
selected_segmentation_df = pd.DataFrame([
    {
        "role": "main",
        "config": "w2_o0p5",
        "window_sec": 2.0,
        "overlap": 0.5,
        "step_sec": 1.0,
        "reason": "Chosen as the main segmentation setting for the privacy-utility study.",
    },
    {
        "role": "comparison",
        "config": "w3_o0p5",
        "window_sec": 3.0,
        "overlap": 0.5,
        "step_sec": 1.5,
        "reason": "Kept as a higher-utility comparison setting.",
    },
])

selected_segmentation_df


,role,config,window_sec,overlap,step_sec,reason
0,main,w2_o0p5,2.0,0.5,1.0,Chosen as the main segmentation setting for th...
1,comparison,w3_o0p5,3.0,0.5,1.5,Kept as a higher-utility comparison setting.


## Reproducibility Note

The raw sweep summaries are stored in:
- [outputs/tables/baseline_sweep_summary.csv](/c:/Users/Tiago/Documents/GitHub/ecg-privacy/outputs/tables/baseline_sweep_summary.csv)

The derived datasets used during this study are stored in:
- [data/interim/window_overlap_sweep](/c:/Users/Tiago/Documents/GitHub/ecg-privacy/data/interim/window_overlap_sweep)
